# nb62 - Depth and iterative refinement: the last two architecture gaps (H24)

**Context.** Architecture levers are 0-for-8 on this data while every real gain came from objective/training. Two claims remain unmeasured under the champion recipe and are closed here.

**Hypotheses (anchor = nb58 qd+EMA singles 0.0418 +/- 0.0005; win = >0.002 overall or any E>17 bin; 2 seeds; recipe otherwise identical).**
- H24a `deep6`: depth L=3 -> 6 at fixed d=128. The old capacity scan (nb22) predates quant/EMA/qd; width (d256) was flat but DEPTH under the champion recipe was never measured.
- H24b `refine2`: two-pass iterative refinement - pass 1 computes the signal gate, the residual map (1-w)*e is appended as a token feature, and a second encoder re-reads the window with the residual visible before the final quantiles. This is the end-to-end learnable version of the Lednev subtract-and-refit loop (unlike H23c, which only received a frozen fit's output).

**Stakes stated up front.** If either wins, architecture re-enters the game. If both are flat, the information-bound conclusion (remaining ~0.010 = in-core photonic overlap ambiguity, unresolvable from window contents) closes with no remaining loopholes.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, prep
from picocal_models import QUANTILES, width_binned_calibration, CFG, qd_pinball_loss
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB62_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB62_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(D['Eraw']).to(DEVICE))
ktr, kva, kte, ctr = D['ktr'], D['kva'], D['kte'], D['ctr']
y = D['y']; Et = D['Et']
QS = torch.tensor(QUANTILES, device=DEVICE)
NG = 6
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 147s


In [2]:
def make_encoder(d, layers):
    layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                       dropout=CFG['dropout'], batch_first=True)
    return nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
class SubNetDeep(nn.Module):
    def __init__(self, in_dim, la0, lb0, layers):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        self.enc = make_encoder(d, layers)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
class SubNetRefine(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed1 = nn.Linear(in_dim, d)
        self.enc1 = make_encoder(d, 2)
        self.fhead1 = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.embed2 = nn.Linear(in_dim + 1, d)
        self.enc2 = make_encoder(d, 3)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead2 = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h1 = self.enc1(self.embed1(x), src_key_padding_mask=~m)
        w1 = torch.sigmoid(self.fhead1(h1).squeeze(-1)) * m.float()
        resid = torch.log1p(((1.0 - w1) * ecell).clamp(min=0))
        x2 = torch.cat([x, resid.unsqueeze(-1)], -1)
        h2 = self.enc2(self.embed2(x2), src_key_padding_mask=~m)
        w2 = torch.sigmoid(self.fhead2(h2).squeeze(-1)) * m.float()
        base = self.la * torch.log1p((w2 * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h2 * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
def build_model(config):
    if config == 'deep6':
        return SubNetDeep(D['IN_DIM'], D['la0'], D['lb0'], layers=6)
    return SubNetRefine(D['IN_DIM'], D['la0'], D['lb0'])
def train_eval(config, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = build_model(config).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb62_{config}_s{seed}.pt'
    def batches(idx, bs, sh=None):
        idx = np.asarray(idx)
        if sh is not None: idx = sh.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(m_, b): return m_(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def vloss(m_):
        m_.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256):
                d = T['Y'][b] - fwd(m_, b)
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from ep {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            opt.zero_grad()
            qd_pinball_loss(fwd(model, b), T['Y'][b], QS).backward()
            opt.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = build_model(config).to(DEVICE)
    final.load_state_dict(bstate); final.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256): out.append(fwd(final, b).cpu().numpy())
        return np.concatenate(out)
    pe = width_binned_calibration(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [3]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
JOBS = {'smoke': [('deep6', 0), ('refine2', 0)],
        'full': [(cfg, s) for cfg in ('deep6', 'refine2') for s in (0, 1)]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb62_arch{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
for config, seed in JOBS:
    if (config, seed) in done: print('skip', config, seed); continue
    t1 = time.time()
    sig, pe = train_eval(config, seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb62_pred{TAG}_{config}_s{seed}.npy', pe)
    row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'{config} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

deep6 seed 0: sigma_eff 0.0426 (4535s)


deep6 seed 1: sigma_eff 0.0424 (4397s)


refine2 seed 0: sigma_eff 0.0415 (3763s)


refine2 seed 1: sigma_eff 0.0419 (4419s)


 config  seed  sigma_eff  elapsed
  deep6     0     0.0426     4535
  deep6     1     0.0424     4397
refine2     0     0.0415     3763
refine2     1     0.0419     4419


## Verdict

Anchor: nb58 qd+EMA singles 0.0418 +/- 0.0005 (L=3, single pass, otherwise identical). Win = >0.002 overall or any E>17 bin.

In [4]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
print('anchor: nb58 qd+EMA singles 0.0418 +/- 0.0005 | stack record 0.0402 (LS 0.0411)')
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
for cfg in ('deep6', 'refine2'):
    preds = [np.load(OUT / f'nb62_pred{TAG}_{cfg}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb62_pred{TAG}_{cfg}_s{s}.npy').exists()]
    if not preds: continue
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    bins = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        bins.append(f'{resolution(ens[mm], te_e[mm])["sigma_eff"]:.4f}')
    print(f'{cfg:8s} mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('         per-bin ' + ' / '.join(bins))

anchor: nb58 qd+EMA singles 0.0418 +/- 0.0005 | stack record 0.0402 (LS 0.0411)
deep6    mean 0.0425 +/- 0.0001 | ens 0.0419
         per-bin 0.0655 / 0.0464 / 0.0349 / 0.0356 / 0.0332 / 0.0360
refine2  mean 0.0417 +/- 0.0002 | ens 0.0409
         per-bin 0.0631 / 0.0464 / 0.0340 / 0.0352 / 0.0339 / 0.0353
